In [46]:
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder
import joblib

# --- 1. Загрузка ваших данных ---
# Замените 'your_data.csv' на имя вашего файла
# Убедитесь, что в скобках указано точное название колонки с данными
df = pd.read_csv('csv_data/train.csv') 
column_name = 'class_label'  # <--- ВПИШИТЕ СЮДА НАЗВАНИЕ КОЛОНКИ

# --- 2. Очистка данных (удаление хеша) ---
# Удаляем все символы в начале строки до первой заглавной буквы
df['clean_name'] = df[column_name].str.replace(r'^[^A-Z]*', '', regex=True)

# --- 3. Преобразование в числа (0, 1, 2...) ---
le = LabelEncoder()
df['category_id'] = le.fit_transform(df['clean_name'])

# --- 4. Сохранение обработанных данных в CSV ---
df.to_csv('processed_data.csv', index=False, encoding='utf-8-sig')
print("✅ Данные сохранены в файл: processed_data.csv")

# --- 5. Сохранение справочника соответствия (Класс <-> Число) ---
mapping_df = pd.DataFrame({
    'category_id': range(len(le.classes_)),
    'planet_name': le.classes_
})
mapping_df.to_csv('class_mapping.csv', index=False, encoding='utf-8-sig')
print("✅ Справочник сохранен в файл: class_mapping.csv")

# --- 6. Сохранение кодировщика (для использования в будущем) ---
joblib.dump(le, 'planet_encoder.pkl')
print("✅ Кодировщик сохранен в файл: planet_encoder.pkl")

# --- Вывод превью результата в консоль ---
print("\n👀 Первые 5 строк результата:")
print(df[[column_name, 'clean_name', 'category_id']].head())

✅ Данные сохранены в файл: processed_data.csv
✅ Справочник сохранен в файл: class_mapping.csv
✅ Кодировщик сохранен в файл: planet_encoder.pkl

👀 Первые 5 строк результата:
                                    class_label   clean_name  category_id
0       ee9f167dba3861c8f5b5bdbcb9b84b86K2-155d      K2-155d            7
1    f5fcfa4be3a8c036be1fde7432958048Kepler-62f   Kepler-62f           19
2       4bf780ec00b0bb3a4e4c0cfe8ea68559Gliese_      Gliese_            1
3   cd304eb2d58ec264e73d4a187cddf052Kepler-296f  Kepler-296f           17
4  cd8e353cb4d5151aefb9e6cd1fa7221255_Cancri_Bc    Cancri_Bc            0


In [ ]:
df = pd.read_csv("processed_data.csv")


df = df.drop(columns="audio_signal")
df.to_csv("popka.csv")

In [94]:
import pandas as pd
import plotly.graph_objects as go

def plot_category_distribution(
    dataframe: pd.DataFrame, 
    column: str, 
    title: str = None,
    color: str = "#00BCD4",
    bg_color: str = "#FFFFFF",  # 🎨 Цвет фона (по умолчанию белый)
    save_path: str = None
) -> go.Figure:
    
    if column not in dataframe.columns:
        raise ValueError(f"Колонка '{column}' не найдена в DataFrame")

    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=dataframe[column],
        name=column,
        marker_color=color,
        opacity=0.75,
        hovertemplate='<b>Категория:</b> %{x}<br><b>Частота:</b> %{y}<extra></extra>'
    ))

    # Настройка оси X (все подписи)
    min_val = int(dataframe[column].min())
    max_val = int(dataframe[column].max())
    
    fig.update_xaxes(
        tickmode='linear',
        dtick=1,
        range=[min_val - 0.5, max_val + 0.5],
        tickangle=45,
        tickfont=dict(size=10)
    )

    # 🎨 НАСТРОЙКА ЦВЕТОВ И ФОНА
    fig.update_layout(
        title={
            'text': title or f"Распределение категорий в тесте",
            'y':0.95,
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(size=24, family="Arial Black", color="#00BCD4")
        },
        xaxis_title="ID Категории",
        yaxis_title="Количество записей",
        
        # --- ЦВЕТА ФОНА ---
        paper_bgcolor=bg_color,  # Внешний фон
        plot_bgcolor=bg_color,   # Внутренний фон (область графика)
        
        # Цвет сетки и осей (чтобы сочетались с фоном)
        xaxis=dict(showgrid=False, gridcolor="#1DAAE6", linecolor="#00BCD4"),
        yaxis=dict(showgrid=True, gridcolor="#1DAAE6", linecolor="#00BCD4"),
        
        hovermode="x",
        bargap=0.1,
        height=600,
        width=1000,
        font=dict(family="Roboto, sans-serif", size=14, color="#00BCD4")
    )

    if save_path:
        fig.write_html(save_path)
        print(f"График сохранен в {save_path}")

    return fig

# --- Примеры использования ---

df = pd.read_csv("popka.csv")

# 1. Белый фон (классика)
fig1 = plot_category_distribution(df, "category_id", bg_color="#000066")
fig1.show()


In [96]:
import pandas as pd
import plotly.graph_objects as go

def plot_category_distribution(
    dataframe: pd.DataFrame, 
    column: str, 
    title: str = None,
    color: str = "#00BCD4",
    bg_color: str = "#FFFFFF",
    top_n: int = None,      # 🆕 Параметр: сколько топ-категорий показать
    save_path: str = None
) -> go.Figure:
    
    if column not in dataframe.columns:
        raise ValueError(f"Колонка '{column}' не найдена в DataFrame")

    fig = go.Figure()

    # 🆕 ЛОГИКА: Гистограмма или Топ-N?
    if top_n is None:
        # --- Режим гистограммы (все данные) ---
        fig.add_trace(go.Histogram(
            x=dataframe[column],
            name=column,
            marker_color=color,
            opacity=0.75,
            hovertemplate='<b>Категория:</b> %{x}<br><b>Частота:</b> %{y}<extra></extra>'
        ))
        
        # Настройка оси X для гистограммы (все подписи)
        min_val = int(dataframe[column].min())
        max_val = int(dataframe[column].max())
        fig.update_xaxes(
            tickmode='linear',
            dtick=1,
            range=[min_val - 0.5, max_val + 0.5],
            tickangle=45,
            tickfont=dict(size=10)
        )
        bargap = 0.1
        
    else:
        # --- Режим ТОП-N (Bar Chart) ---
        # Считаем количество вхождений и берем топ-N
        top_data = dataframe[column].value_counts().head(top_n)
        
        fig.add_trace(go.Bar(
            x=top_data.index.astype(str),  # Категории как строки
            y=top_data.values,
            name=f"Top {top_n}",
            marker_color=color,
            opacity=0.85,
            hovertemplate='<b>Категория:</b> %{x}<br><b>Частота:</b> %{y}<extra></extra>'
        ))
        
        # Для бара подписи не нужно поворачивать, если их мало
        fig.update_xaxes(
            tickangle=0,
            tickfont=dict(size=12)
        )
        bargap = 0.3  # Для бара зазор можно сделать больше для красоты

    # 🎨 НАСТРОЙКА ЦВЕТОВ И ФОНА
    fig.update_layout(
        title={
            'text': title or (f"Топ-{top_n} категорий" if top_n else "Распределение категорий в тесте"),
            'y':0.95,
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(size=24, family="Arial Black", color="#00BCD4")
        },
        xaxis_title="ID Категории",
        yaxis_title="Количество записей",
        
        # --- ЦВЕТА ФОНА ---
        paper_bgcolor=bg_color,
        plot_bgcolor=bg_color,
        
        # Цвет сетки и осей
        xaxis=dict(showgrid=False, gridcolor="#1DAAE6", linecolor="#00BCD4"),
        yaxis=dict(showgrid=True, gridcolor="#1DAAE6", linecolor="#00BCD4"),
        
        hovermode="x",
        bargap=bargap,
        height=600,
        width=1000,
        font=dict(family="Roboto, sans-serif", size=14, color="#00BCD4")
    )

    if save_path:
        fig.write_html(save_path)
        print(f"График сохранен в {save_path}")

    return fig

# --- Примеры использования ---

df = pd.read_csv("popka.csv")


# 2. 🆕 ТОП-5 категорий (темно-синий фон)
fig2 = plot_category_distribution(
    df, 
    "category_id", 
    bg_color="#000066",
    top_n=5,  # 🔥 Только топ-5
    title="Топ-5 самых популярных категорий"
)
fig2.show()

In [93]:
import pandas as pd
import plotly.graph_objects as go

# 1. Подготовка данных (берем первые 100 строк)
subset_data = df.iloc[:100]
y_values = subset_data['category_id']
x_values = list(range(len(y_values)))  # Номера строк: 0, 1, 2...

# 2. Создаем фигуру
fig = go.Figure()

fig.add_trace(go.Bar(
    x=x_values,
    y=y_values,
    marker_color="#00BCD4",      # Голубой цвет столбцов
    marker_line_color="#000066", # Обводка (для контраста с фоном)
    marker_line_width=1.5,
    opacity=0.85,
    hovertemplate='<b>Строка:</b> %{x}<br><b>Значение:</b> %{y}<extra></extra>'
))

# 3. Настройка макета (в стиле "идеального кода")
fig.update_layout(
    title={
        'text': "График по первым 100 строкам",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=24, family="Arial Black", color="#00BCD4")
    },
    xaxis_title="Номер строки",
    yaxis_title="Значение колонки",
    
    # --- ЦВЕТА ФОНА ---
    paper_bgcolor="#000066",  # Темно-синий фон
    plot_bgcolor="#000066",
    
    # --- СЕТКА И ОСИ ---
    xaxis=dict(
        showgrid=False, 
        gridcolor="#1DAAE6", 
        linecolor="#00BCD4",
        tickfont=dict(color="#00BCD4")
    ),
    yaxis=dict(
        showgrid=True, 
        gridcolor="#1DAAE6", 
        linecolor="#00BCD4",
        tickfont=dict(color="#00BCD4")
    ),
    
    # --- ТОЛЩИНА КОЛОНОК ---
    bargap=0.5,        # 🎯 Главный параметр: 0.5 = узкие колонки (0.0 = широкие)
    bargroupgap=0.001,
    
    hovermode="x",
    height=500,
    width=1000,
    font=dict(family="Roboto, sans-serif", size=14, color="#00BCD4")
)

fig.show()

In [97]:
import pandas as pd

# 1. Читаем ваш CSV файл
df = pd.read_csv('popka.csv')

# 2. Берем последний символ из колонки 'class_label' и кладем в 'category_id'
# .str[-1] берет последний символ строки
df['category_id'] = df['class_label'].astype(str).str[-1]

# 3. Сохраняем результат в новый файл (или перезаписываем старый)
# index=False нужен, чтобы не создавать лишний столбец с номерами строк при сохранении
df.to_csv('popka_updated.csv', index=False)

print("Готово! Проверьте файл popka_updated.csv")

Готово! Проверьте файл popka_updated.csv


In [99]:
import pandas as pd
import numpy as np
import hashlib

def parse_audio_signal(signal_str):
    """
    Превращает строку вида "-8.43e-07,8.42e-07,..." в numpy массив.
    """
    # Убираем скобки, если они есть, и лишние пробелы
    signal_str = signal_str.strip().strip('()').strip('[]')
    # Разделяем по запятой и конвертируем в float
    values = [float(x.strip()) for x in signal_str.split(',') if x.strip()]
    return np.array(values, dtype=np.float32)

def get_audio_hash(audio_array, tolerance=6):
    """
    Вычисляет хеш для аудио-сигнала.
    tolerance=6 означает округление до 6 знаков после запятой,
    чтобы избежать проблем с плавающей точкой.
    """
    # Округляем значения, чтобы微小шие различия в float не создавали новые классы
    rounded = np.round(audio_array, decimals=tolerance)
    # Превращаем в байты
    data_bytes = rounded.tobytes()
    # Вычисляем MD5 хеш
    return hashlib.md5(data_bytes).hexdigest()

def restore_labels(input_file='csv_data/train.csv', output_file='train_restored.csv'):
    print(f"Чтение файла {input_file}...")
    df = pd.read_csv(input_file)
    
    print(f"Всего строк: {len(df)}")
    print(f"Уникальных текстовых меток: {df['class_label'].nunique()}")
    
    # Словарь для маппинга: хеш_звука -> номер_класса
    sound_hash_to_class = {}
    current_class_id = 0
    
    # Список для новых меток
    new_labels = []
    
    print("Обработка и восстановление меток...")
    
    for idx, row in df.iterrows():
        # 1. Парсим аудиосигнал из строки
        audio_array = parse_audio_signal(row['audio_signal'])
        
        # 2. Получаем хеш содержимого звука
        sound_hash = get_audio_hash(audio_array)
        
        # 3. Если этот звук встретился впервые, присваиваем новый класс
        if sound_hash not in sound_hash_to_class:
            sound_hash_to_class[sound_hash] = current_class_id
            current_class_id += 1
        
        # 4. Получаем номер класса для этой строки
        class_id = sound_hash_to_class[sound_hash]
        new_labels.append(class_id)
    
    # Заменяем колонку class_label на восстановленные целые числа
    df['class_label'] = new_labels
    
    # Сохраняем результат
    df.to_csv(output_file, index=False)
    
    # Статистика
    unique_classes = len(sound_hash_to_class)
    print(f"\n=== Результаты ===")
    print(f"Уникальных звуков (классов) найдено: {unique_classes}")
    print(f"Распределение классов: {df['class_label'].value_counts().sort_index().to_dict()}")
    print(f"Файл сохранен: {output_file}")
    
    # Показываем пример соответствия (первые 10 строк)
    print(f"\n=== Пример первых 10 строк ===")
    print(df[['id', 'class_label']].head(10).to_string(index=False))

if __name__ == '__main__':
    restore_labels()

Чтение файла csv_data/train.csv...
Всего строк: 1200
Уникальных текстовых меток: 1163
Обработка и восстановление меток...

=== Результаты ===
Уникальных звуков (классов) найдено: 1200
Распределение классов: {0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 1, 18: 1, 19: 1, 20: 1, 21: 1, 22: 1, 23: 1, 24: 1, 25: 1, 26: 1, 27: 1, 28: 1, 29: 1, 30: 1, 31: 1, 32: 1, 33: 1, 34: 1, 35: 1, 36: 1, 37: 1, 38: 1, 39: 1, 40: 1, 41: 1, 42: 1, 43: 1, 44: 1, 45: 1, 46: 1, 47: 1, 48: 1, 49: 1, 50: 1, 51: 1, 52: 1, 53: 1, 54: 1, 55: 1, 56: 1, 57: 1, 58: 1, 59: 1, 60: 1, 61: 1, 62: 1, 63: 1, 64: 1, 65: 1, 66: 1, 67: 1, 68: 1, 69: 1, 70: 1, 71: 1, 72: 1, 73: 1, 74: 1, 75: 1, 76: 1, 77: 1, 78: 1, 79: 1, 80: 1, 81: 1, 82: 1, 83: 1, 84: 1, 85: 1, 86: 1, 87: 1, 88: 1, 89: 1, 90: 1, 91: 1, 92: 1, 93: 1, 94: 1, 95: 1, 96: 1, 97: 1, 98: 1, 99: 1, 100: 1, 101: 1, 102: 1, 103: 1, 104: 1, 105: 1, 106: 1, 107: 1, 108: 1, 109: 1, 110: 1, 111: 1, 112: 1